# Observe 模块教程

本教程介绍 open-xquant 的可观测性层：**执行追踪**、**审计记录**、**实验日志关联**。

策略运行不应该是黑箱。Observe 模块提供三层可追溯体系，从高层决策到底层组件执行细节，层层可查：

```
ExperimentLog         "改了什么 → 效果如何"（决策层面）
    │ run_id
    ▼
AuditRecord           "完整配置 + 结果指纹"（可复现层面）
    │ trace_spans
    ▼
[TraceSpan, ...]      "每个组件的输入输出"（调试层面）
```

**核心类型**：

| 类 | 职责 |
|---|---|
| `TraceSpan` | 单个组件（Indicator/Signal/Rule）的执行记录 |
| `DefaultTracer` | 收集 TraceSpan 的回调实现，传入 Engine |
| `AuditRecord` | 一次完整运行的审计快照（配置 + 追踪 + 分层哈希） |
| `ExperimentLog` | 策略迭代实验日志，通过 `run_id` 关联 AuditRecord |
| `StrategyMonitor` | 策略健康度监控（滚动 Sharpe、回撤、异常期间检测） |
| `MarketStateDetector` | 市场状态分类（高/正常/低波动率） |

---
## 1. 准备工作：构建一个策略并回测

先快速搭建一个 SMA 交叉策略，作为后续 observe 演示的基础。

In [ ]:
from oxq.core import Engine, Strategy
from oxq.data import LocalMarketDataProvider
from oxq.indicators import SMA
from oxq.signals import Crossover
from oxq.rules import ExitRule
from oxq.trade import SimBroker
from oxq.universe import StaticUniverse
from oxq.portfolio.optimizers import EqualWeightOptimizer

# 创建 Crossover 信号，并声明它依赖的 Indicator
crossover = Crossover()
crossover.required_indicators = {
    "sma_10": (SMA(), {"period": 10}),
    "sma_50": (SMA(), {"period": 50}),
}

strategy = Strategy(
    name="sma_crossover",
    hypothesis="短期均线上穿长期均线的标的在后续持有期内有正超额收益",
    objectives={
        "total_return": {"min": 0.05},
        "sharpe_ratio": {"min": 0.5},
        "max_drawdown": {"max": -0.25},
    },
    benchmarks=["SPY"],
    universe=StaticUniverse(("AAPL",)),
    signals={
        "golden_cross": (crossover, {"fast": "sma_10", "slow": "sma_50"}),
    },
    portfolio=EqualWeightOptimizer(),
)

print(f"策略: {strategy.name}")
print(f"假设: {strategy.hypothesis}")
print(f"信号: {list(strategy.signals.keys())}")
print(f"组合优化器: {strategy.portfolio.name}")

---
## 2. 使用 DefaultTracer 追踪执行过程

`DefaultTracer` 是 Engine 的可选回调对象。传入后，Engine 会在每个组件（Indicator、Signal、Rule）执行时自动记录 `TraceSpan`。

不传 tracer 时完全无开销——只是一个 `if tracer:` 判空。

In [ ]:
from oxq.observe import DefaultTracer

# 创建 tracer 并传入 engine.run()
tracer = DefaultTracer()

# Rule 在 engine.run() 时传入，不属于 Strategy
exit_rule = ExitRule(fast="sma_10", slow="sma_50")

engine = Engine()
result = engine.run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=SimBroker(),
    rules=[exit_rule],  # ← Rule 通过 rules 参数传入
    start="2023-01-01",
    end="2024-12-31",
    tracer=tracer,  # ← 只需加这一个参数
)

print(f"总收益率:   {result.total_return():.2%}")
print(f"Sharpe:     {result.sharpe_ratio():.2f}")
print(f"最大回撤:   {result.max_drawdown():.2%}")
print(f"交易次数:   {len(result.trades)}")
print(f"追踪到的 Span 数: {len(tracer.spans)}")

### 查看 TraceSpan 详情

每个 TraceSpan 记录了一个组件的执行信息：组件名、输入参数、输出摘要、耗时、状态。

In [ ]:
for span in tracer.spans:
    print(f"[{span.status}] {span.component}")
    print(f"  输入: {span.inputs}")
    print(f"  输出: {span.output_summary}")
    print(f"  耗时: {span.duration_ms:.2f} ms")
    print()

可以看到 Engine 依次执行了：
1. **indicator:sma_10** — 计算 10 日均线（从 Signal 的 `required_indicators` 自动收集）
2. **indicator:sma_50** — 计算 50 日均线
3. **signal:golden_cross** — 检测金叉信号
4. **rule:ExitRule** — 出场规则（通过 `rules` 参数传入）

PortfolioOptimizer 负责根据信号计算目标权重，Rule 负责逐 bar 评估退出条件。每个 Span 都有输入参数、输出摘要和精确耗时，可以直接定位哪个组件有问题。

---
## 3. 生成 AuditRecord — 运行的完整档案

`AuditRecord` 是一次运行的完整审计快照，包含：
- 策略配置（用了哪些指标、什么参数、什么规则）
- 所有 TraceSpan（执行过程）
- **分层哈希**（mktdata_hash + trades_hash + equity_hash → result_hash）

分层哈希的价值：用相同配置重跑，比对 hash 即可验证确定性。hash 不一致时，可以直接看是哪一层变了。

In [ ]:
from oxq.observe import AuditRecord

audit = AuditRecord.build(
    tracer=tracer,
    result=result,
    strategy_name=strategy.name,
    strategy_config={
        "signals": {k: v[1] for k, v in strategy.signals.items()},
        "portfolio": strategy.portfolio.name,
    },
    start_date="2023-01-01",
    end_date="2024-12-31",
    initial_cash=100_000.0,
)

print(f"Run ID:       {audit.run_id}")
print(f"策略:          {audit.strategy_name}")
print(f"时间范围:      {audit.start_date} ~ {audit.end_date}")
print(f"TraceSpan 数:  {len(audit.trace_spans)}")
print()
print("分层哈希:")
print(f"  mktdata:  {audit.mktdata_hash[:30]}...")
print(f"  trades:   {audit.trades_hash[:30]}...")
print(f"  equity:   {audit.equity_hash[:30]}...")
print(f"  combined: {audit.result_hash[:30]}...")

### 查看策略配置快照

AuditRecord 保存了完整的策略参数，即使事后修改了代码，也能追溯当时的配置。

In [ ]:
import json

print("策略配置快照:")
print(json.dumps(audit.strategy_config, indent=2))

### 持久化为 JSON 文件

AuditRecord 支持 JSON 序列化，可以保存到文件系统，事后加载。

In [ ]:
import tempfile
from pathlib import Path

# 保存到临时目录演示
tmpdir = tempfile.mkdtemp()
audit_path = Path(tmpdir) / f"{audit.run_id}.json"

audit.to_json(str(audit_path))
print(f"已保存到: {audit_path}")
print(f"文件大小: {audit_path.stat().st_size:,} 字节")

# 加载回来
restored = AuditRecord.from_json(str(audit_path))
print(f"\n加载成功: {restored.run_id}")
print(f"哈希一致: {restored.result_hash == audit.result_hash}")

---
## 4. 验证确定性 — 重跑并比对哈希

框架的核心原则之一是**确定性**：相同输入必须产生相同输出。分层哈希让你可以验证这一点。

In [ ]:
# 用完全相同的配置重跑一次
tracer2 = DefaultTracer()
result2 = Engine().run(
    strategy,
    market=LocalMarketDataProvider(),
    broker=SimBroker(),
    rules=[ExitRule(fast="sma_10", slow="sma_50")],
    start="2023-01-01",
    end="2024-12-31",
    tracer=tracer2,
)

audit2 = AuditRecord.build(
    tracer=tracer2, result=result2,
    strategy_name=strategy.name,
    strategy_config=audit.strategy_config,
    start_date="2023-01-01", end_date="2024-12-31",
    initial_cash=100_000.0,
)

# 逐层比对哈希
print("确定性验证:")
print(f"  mktdata 哈希一致: {audit.mktdata_hash == audit2.mktdata_hash}")
print(f"  trades  哈希一致: {audit.trades_hash == audit2.trades_hash}")
print(f"  equity  哈希一致: {audit.equity_hash == audit2.equity_hash}")
print(f"  总哈希一致:       {audit.result_hash == audit2.result_hash}")

所有哈希完全一致——这证明了引擎的确定性。如果某天重跑后 trades_hash 变了但 mktdata_hash 没变，就说明问题出在 Rule 阶段（订单生成逻辑），而不是 Indicator 计算。这就是分层哈希的定位价值。

---
## 5. ExperimentLog — 策略迭代的实验日志

在实际量化研究中，你会不断迭代策略参数。`ExperimentLog` 记录每次迭代的观察、假设、标准、结果和结论。

通过 `run_id` 字段，每条实验记录都可以关联到一个 `AuditRecord`，实现从"改了什么"追溯到"完整执行细节"。

In [ ]:
from oxq.observe import ExperimentLog

log = ExperimentLog(name="sma_crossover_iterations")

# 第一轮：baseline SMA(10, 50)
log.add_from_strategy(
    strategy=strategy,
    result=result,
    observation="Baseline 回测，SMA(10, 50)",
    conclusion="Sharpe 偏低，尝试调整快线周期",
    run_id=audit.run_id,  # ← 关联到 AuditRecord
)

print(f"实验日志: {log.name}")
print(f"记录数:   {len(log.experiments)}")
print()

exp = log.experiments[0]
print(f"名称:     {exp.name}")
print(f"观察:     {exp.observation}")
print(f"假设:     {exp.hypothesis}")
print(f"结论:     {exp.conclusion}")
print(f"run_id:   {exp.run_id}")
print(f"指标:     {exp.result}")

### 第二轮迭代：调整参数

把快线从 SMA(10) 改为 SMA(20)，重新回测并记录。

In [ ]:
# 第二轮：SMA(20, 50)
crossover_v2 = Crossover()
crossover_v2.required_indicators = {
    "sma_20": (SMA(), {"period": 20}),
    "sma_50": (SMA(), {"period": 50}),
}

strategy_v2 = Strategy(
    name="sma_crossover",
    hypothesis="短期均线上穿长期均线的标的在后续持有期内有正超额收益",
    objectives=strategy.objectives,
    benchmarks=["SPY"],
    universe=StaticUniverse(("AAPL",)),
    signals={
        "golden_cross": (crossover_v2, {"fast": "sma_20", "slow": "sma_50"}),
    },
    portfolio=EqualWeightOptimizer(),
)

tracer_v2 = DefaultTracer()
result_v2 = Engine().run(
    strategy_v2,
    market=LocalMarketDataProvider(),
    broker=SimBroker(),
    rules=[ExitRule(fast="sma_20", slow="sma_50")],
    start="2023-01-01",
    end="2024-12-31",
    tracer=tracer_v2,
)

audit_v2 = AuditRecord.build(
    tracer=tracer_v2, result=result_v2,
    strategy_name=strategy_v2.name,
    strategy_config={
        "signals": {k: v[1] for k, v in strategy_v2.signals.items()},
        "portfolio": strategy_v2.portfolio.name,
    },
    start_date="2023-01-01", end_date="2024-12-31",
    initial_cash=100_000.0,
)
audit_v2.to_json(str(Path(tmpdir) / f"{audit_v2.run_id}.json"))

log.add_from_strategy(
    strategy=strategy_v2,
    result=result_v2,
    observation="快线从 SMA(10) 改为 SMA(20)",
    conclusion="Sharpe 有变化，记录对比",
    run_id=audit_v2.run_id,
)

print(f"V1 总收益: {result.total_return():.2%}  Sharpe: {result.sharpe_ratio():.2f}")
print(f"V2 总收益: {result_v2.total_return():.2%}  Sharpe: {result_v2.sharpe_ratio():.2f}")

### 查看实验日志全景

`to_dataframe()` 将所有实验记录转为 DataFrame，一目了然。

In [ ]:
df_log = log.to_dataframe()
df_log[["name", "observation", "conclusion", "run_id"]]

### 从实验日志追溯到审计详情

通过 `run_id` 加载对应的 AuditRecord，对比两次运行的配置差异。

In [ ]:
# 从实验日志拿到两次运行的 run_id
run_id_v1 = log.experiments[0].run_id
run_id_v2 = log.experiments[1].run_id

# 加载两份审计记录
a1 = AuditRecord.from_json(str(Path(tmpdir) / f"{run_id_v1}.json"))
a2 = AuditRecord.from_json(str(Path(tmpdir) / f"{run_id_v2}.json"))

# 对比配置
print("V1 信号配置:", a1.strategy_config["signals"])
print("V2 信号配置:", a2.strategy_config["signals"])
print()

# 对比哈希 — 信号参数变了，所以 mktdata 和 trades 都会不同
print(f"mktdata 相同: {a1.mktdata_hash == a2.mktdata_hash}")
print(f"trades  相同: {a1.trades_hash == a2.trades_hash}")
print(f"equity  相同: {a1.equity_hash == a2.equity_hash}")

---
## 6. StrategyMonitor — 策略健康度监控

`StrategyMonitor` 消费一个 `RunResult`，计算滚动 Sharpe、滚动回撤、并自动检测策略表现恶化的"坏周期"（rolling Sharpe < 0 持续超过阈值天数）。

In [ ]:
from oxq.observe import StrategyMonitor

monitor = StrategyMonitor(result, roll_window=63, min_bad_days=20)

# 策略健康摘要
summary = monitor.summary()
print("策略健康摘要:")
for k, v in summary.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
    else:
        print(f"  {k}: {v}")

In [ ]:
# 检测到的坏周期
if monitor.bad_periods:
    print(f"检测到 {len(monitor.bad_periods)} 个坏周期:\n")
    for bp in monitor.bad_periods:
        print(f"  {bp.start} ~ {bp.end} ({bp.days} 天, 平均 Sharpe: {bp.avg_sharpe:.2f})")
else:
    print("未检测到持续恶化的坏周期")

In [ ]:
# 滚动 Sharpe 和滚动回撤的时间序列
print("滚动 Sharpe (最近 5 个数据点):")
print(monitor.rolling_sharpe.dropna().tail())
print()
print("滚动回撤 (最近 5 个数据点):")
print(monitor.rolling_drawdown.tail())

---
## 7. MarketStateDetector — 市场状态分类

`MarketStateDetector` 根据波动率将市场划分为三种状态（高波动/正常/低波动），并可以按状态拆分策略表现——帮你理解策略在不同市场环境下的行为。

In [ ]:
from oxq.observe import MarketStateDetector

detector = MarketStateDetector(result, vol_lookback=20)

# 波动率阈值
print(f"波动率中位数:     {detector.vol_median:.4f}")
print(f"高波动线 (1.3x):  {detector.high_vol_line:.4f}")
print(f"低波动线 (0.7x):  {detector.low_vol_line:.4f}")
print()

# 各状态天数统计
states = detector.states.dropna()
print("市场状态分布:")
print(states.value_counts().to_string())

In [ ]:
# 按市场状态拆分策略表现
perf = detector.performance_by_state(result)

print("按市场状态拆分的策略表现:")
print(f"{'状态':<8} {'天数':>6} {'年化收益':>10} {'Sharpe':>8}")
print("-" * 36)
for state in ("high", "normal", "low"):
    if state in perf:
        p = perf[state]
        print(f"{state:<8} {p['days']:>6} {p['ann_return']:>10.2%} {p['sharpe']:>8.2f}")

---
## 小结

本教程覆盖了 observe 模块的核心用法：

### 三层可追溯体系

| 层级 | 类 | 回答的问题 |
|------|---|-----------|
| 决策层 | `ExperimentLog` | 改了什么？效果如何？ |
| 复现层 | `AuditRecord` | 完整配置是什么？结果能复现吗？ |
| 调试层 | `TraceSpan` / `DefaultTracer` | 每个组件的输入输出是什么？ |

### 使用流程

```python
# 1. 创建 tracer，传入 Engine
tracer = DefaultTracer()
result = engine.run(strategy, ..., rules=[exit_rule], tracer=tracer)

# 2. 生成审计记录
audit = AuditRecord.build(tracer=tracer, result=result, ...)
audit.to_json("runs/run_xxx.json")

# 3. 记录到实验日志，关联 run_id
log.add_from_strategy(strategy, result, ..., run_id=audit.run_id)

# 4. 事后追溯：实验日志 → 审计记录 → 组件执行细节
audit = AuditRecord.from_json(f"runs/{exp.run_id}.json")
for span in audit.trace_spans:
    print(span.component, span.output_summary)
```

### 策略诊断

| 类 | 职责 |
|---|---|
| `StrategyMonitor` | 滚动 Sharpe / 回撤 / 坏周期检测 |
| `MarketStateDetector` | 高/正常/低波动率分类 + 按状态拆分表现 |

### 核心设计原则

- **可选零开销** — 不传 tracer 时引擎行为完全不变
- **分层哈希** — 验证失败时可定位到具体哪一层（指标/交易/收益）变了
- **JSON 持久化** — 人可读，Agent 可解析，与项目已有模式一致
- **松耦合** — ExperimentLog 通过 run_id 引用 AuditRecord，三层各自独立存储
- **策略与规则分离** — Strategy 包含 Universe/Signal/PortfolioOptimizer，Rule 在 `engine.run(rules=[...])` 时注入